In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP — B0-only ensemble
# ═══════════════════════════════════════════════════════════════════
import os, sys, time, warnings, glob
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    BASE_DIR    = '/kaggle/input/birdclef-2026'
    # B0 baseline ensemble — use whichever dataset slug holds your 0.882 models
    # (typically /kaggle/input/datasets/brandonkhuu/birdclef-2026-baseline-effb0-onnx
    # or /kaggle/input/datasets/brandonkhuu/birdclef-2026-best-submission, etc.)
    MODEL_DIR   = '/kaggle/input/datasets/brandonkhuu/birdclef-2026-baseline-effb0-onnx'
    TEST_DIR    = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'
    SAMPLE_SUB  = '/kaggle/input/competitions/birdclef-2026/sample_submission.csv'
else:
    BASE_DIR    = 'data/raw'
    MODEL_DIR   = 'experiments'
    TEST_DIR    = os.path.join(BASE_DIR, 'test_soundscapes')
    SAMPLE_SUB  = os.path.join(BASE_DIR, 'sample_submission.csv')

print(f'Environment: {"Kaggle" if ON_KAGGLE else "Local"}')
print(f'Test dir:    {TEST_DIR}')
print(f'Model dir:   {MODEL_DIR}')

# Verify mounted datasets (Kaggle only)
if ON_KAGGLE:
    print(f'\nMounted datasets in /kaggle/input/:')
    try:
        for d in sorted(os.listdir('/kaggle/input')):
            print(f'  {d}')
    except Exception as e:
        print(f'  (could not list: {e})')


In [2]:
# ═══════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES (if needed)
# ═══════════════════════════════════════════════════════════════════
try:
    import onnxruntime as ort
except ImportError:
    import subprocess
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-deps', '-q',
        '/kaggle/input/datasets/brandonkhuu/onnx-runtime-whl/onnxruntime_wheel/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'
    ])
    import onnxruntime as ort

print(f'onnxruntime: {ort.__version__}')

import librosa
print(f'librosa:     {librosa.__version__}')
print(f'numpy:       {np.__version__}')

onnxruntime: 1.24.4
librosa:     0.11.0
numpy:       2.0.2


In [3]:
# ═══════════════════════════════════════════════════════════════════
# SPECTROGRAM CONFIGURATION — matches training pipeline exactly
# ═══════════════════════════════════════════════════════════════════
SAMPLE_RATE      = 32000
N_MELS           = 128
FMAX             = 16000
HOP_LENGTH       = 512
N_FFT            = 2048
WINDOW_SECONDS   = 5.0
WINDOW_SAMPLES   = int(SAMPLE_RATE * WINDOW_SECONDS)  # 160000
NUM_CLASSES      = 234
SPEC_TIME_FRAMES = 313

# Load species list from sample submission (defines column order)
sample_sub = pd.read_csv(SAMPLE_SUB)
SPECIES_LIST = [c for c in sample_sub.columns if c != 'row_id']
assert len(SPECIES_LIST) == NUM_CLASSES, \
    f'Expected {NUM_CLASSES} species, got {len(SPECIES_LIST)}'
print(f'Species: {NUM_CLASSES}')
print(f'Sample submission rows: {len(sample_sub)}')

Species: 234
Sample submission rows: 3


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LOAD ONNX MODELS — B0 ensemble
# ═══════════════════════════════════════════════════════════════════
def find_onnx_models(model_dir, n_folds=5):
    """Find ONNX model files, preferring fp32 over quantized."""
    model_dir = model_dir if isinstance(model_dir, str) else str(model_dir)
    paths = []
    for fold_id in range(n_folds):
        candidates = [
            os.path.join(model_dir, f'baseline_effb0_fold{fold_id}', 'best_model.onnx'),
            os.path.join(model_dir, f'baseline_effb0_fold{fold_id}', 'best_model_quantized.onnx'),
            os.path.join(model_dir, f'fold{fold_id}.onnx'),
            os.path.join(model_dir, f'fold{fold_id}_quantized.onnx'),
            os.path.join(model_dir, f'best_model_fold{fold_id}.onnx'),
            os.path.join(model_dir, f'best_model_fold{fold_id}_quantized.onnx'),
            os.path.join(model_dir, f'best_model_{fold_id}.onnx'),
        ]
        for c in candidates:
            if os.path.exists(c):
                paths.append(c)
                break
    if not paths:
        all_onnx = sorted(glob.glob(os.path.join(model_dir, '**', '*.onnx'), recursive=True))
        regular = [f for f in all_onnx if 'quantized' not in f]
        quantized = [f for f in all_onnx if 'quantized' in f]
        paths = (regular if regular else quantized)[:n_folds]
    return paths

model_paths = find_onnx_models(MODEL_DIR)
print(f'Found {len(model_paths)} ONNX models:')

sessions = []
for p in model_paths:
    sess = ort.InferenceSession(p, providers=['CPUExecutionProvider'])
    sessions.append(sess)
    inp_name = sess.get_inputs()[0].name
    size_mb = os.path.getsize(p) / 1024 / 1024
    print(f'  {os.path.basename(p)}: {size_mb:.1f} MB, input="{inp_name}"')

if len(sessions) == 0:
    raise RuntimeError(
        f'No ONNX models found under {MODEL_DIR!r}. '
        f'Check the dataset is added to the notebook and the path is correct.'
    )

# All B0 models share the same input name; capture once for reuse
INPUT_NAME = sessions[0].get_inputs()[0].name

N_MODELS = len(sessions)
print(f'\nEnsemble size: {N_MODELS} models, input tensor name="{INPUT_NAME}"')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INFERENCE FUNCTIONS — 5× Test-Time Augmentation
#
# Five time-shifted versions of each 5s window:
#   -2.5s, -1.25s, 0, +1.25s, +2.5s
# Aggregation: for each TTA offset, mean logits across folds first; then
# mean across offsets; then sigmoid. This separates the two sources of
# variance (model fold vs time shift) cleanly.
#
# Cost: ~5/3× the inference time of the previous 3× TTA setup. With 5
# B0 fp32 models, roughly 50 min projected. Fits in 80 min budget; the
# existing fallback drops TTA from 5 → 4 → 3 → 2 → 1 if needed.
# ═══════════════════════════════════════════════════════════════════

# 5 offsets evenly spaced over a 5s window (½ window = 2.5s)
TTA_OFFSETS_SAMPLES = [
    -int(0.50 * WINDOW_SAMPLES),  # -2.50 s
    -int(0.25 * WINDOW_SAMPLES),  # -1.25 s
     0,                           #  center
    +int(0.25 * WINDOW_SAMPLES),  # +1.25 s
    +int(0.50 * WINDOW_SAMPLES),  # +2.50 s
]


def compute_melspec(waveform):
    """Log-mel spectrogram matching training pipeline."""
    S = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmax=FMAX,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    if S_db.shape[1] >= SPEC_TIME_FRAMES:
        S_db = S_db[:, :SPEC_TIME_FRAMES]
    else:
        S_db = np.pad(
            S_db, ((0, 0), (0, SPEC_TIME_FRAMES - S_db.shape[1])),
            mode='constant', constant_values=S_db.min(),
        )
    return S_db


def extract_shifted_segment(waveform, center_start, offset_samples):
    """5s segment shifted by offset_samples; zero-pad at boundaries."""
    total_samples = len(waveform)
    start = center_start + offset_samples
    end = start + WINDOW_SAMPLES

    pad_left  = max(0, -start)
    pad_right = max(0, end - total_samples)
    valid_start = max(0, start)
    valid_end   = min(total_samples, end)
    segment = waveform[valid_start:valid_end]

    if pad_left > 0 or pad_right > 0:
        segment = np.pad(segment, (pad_left, pad_right), mode='constant', constant_values=0.0)

    if len(segment) < WINDOW_SAMPLES:
        segment = np.pad(segment, (0, WINDOW_SAMPLES - len(segment)), mode='constant')
    elif len(segment) > WINDOW_SAMPLES:
        segment = segment[:WINDOW_SAMPLES]
    return segment


def _select_offsets(n_tta):
    """
    Pick which subset of TTA_OFFSETS_SAMPLES to use.
    Always includes the center offset (0) for any n_tta >= 1, and prefers
    symmetric outer offsets first because those add the most variance.
    """
    if n_tta >= 5:
        return TTA_OFFSETS_SAMPLES
    if n_tta == 4:
        return [TTA_OFFSETS_SAMPLES[0], TTA_OFFSETS_SAMPLES[1],
                TTA_OFFSETS_SAMPLES[3], TTA_OFFSETS_SAMPLES[4]]  # drop center
    if n_tta == 3:
        return [TTA_OFFSETS_SAMPLES[0], TTA_OFFSETS_SAMPLES[2], TTA_OFFSETS_SAMPLES[4]]
    if n_tta == 2:
        return [TTA_OFFSETS_SAMPLES[2], TTA_OFFSETS_SAMPLES[4]]  # center + +2.5s
    return [TTA_OFFSETS_SAMPLES[2]]  # center only


def predict_window_tta(sessions, waveform, center_start, n_tta=5,
                       input_name=None):
    """
    Per-window prediction.

    Aggregation order:
      1. Per offset: collect logits from all folds, mean → "offset_logits"
      2. Across offsets: mean offset_logits → mean_logits
      3. Sigmoid on mean_logits → final probs

    This is mathematically equivalent to a flat mean across (folds × offsets)
    when N_folds and offsets are constant per call, but more numerically
    stable in mixed-precision and easier to reason about.

    Args:
        sessions: list of ONNX sessions (the fold ensemble)
        waveform: full soundscape audio
        center_start: sample index where this window starts
        n_tta: 1..5 — number of TTA offsets to use (chooses subset)
        input_name: name of the model's input tensor (e.g. "input")

    Returns:
        probs: numpy array of shape (NUM_CLASSES,)
    """
    if input_name is None:
        input_name = INPUT_NAME

    offsets = _select_offsets(n_tta)
    per_offset_logits = []  # one mean-logit-vector per offset

    for offset in offsets:
        segment = extract_shifted_segment(waveform, center_start, offset)
        spec = compute_melspec(segment)
        x = spec[np.newaxis, np.newaxis, :, :].astype(np.float32)

        fold_logits = []
        for sess in sessions:
            logits = sess.run(None, {input_name: x})[0][0]
            fold_logits.append(logits)

        per_offset_logits.append(np.mean(fold_logits, axis=0))

    mean_logits = np.mean(per_offset_logits, axis=0)
    probs = 1.0 / (1.0 + np.exp(-mean_logits))
    return probs


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PROCESS ALL TEST SOUNDSCAPES — 5× TTA with time-budget fallback
#
# Starts at 5× TTA; if projected to overrun the budget, drops to 4×, 3×,
# 2×, or 1× progressively. Better a complete submission than a timeout.
# ═══════════════════════════════════════════════════════════════════

TIME_BUDGET_SECONDS = 80 * 60   # hard ceiling
TIME_WARN_SECONDS   = 75 * 60   # warn at this point
N_TTA_INITIAL = 5               # start with 5× TTA (was 3 in v10)

audio_extensions = ('*.ogg', '*.wav', '*.flac', '*.mp3')
audio_files = []
for ext in audio_extensions:
    audio_files.extend(glob.glob(os.path.join(TEST_DIR, ext)))
audio_files = sorted(audio_files)
print(f'Test soundscapes: {len(audio_files)}')

total_start = time.time()
rows = []
timing = {'audio': 0, 'spec_plus_infer': 0, 'windows': 0}
n_tta_current = N_TTA_INITIAL
tta_downgrades = []

for file_idx, audio_path in enumerate(audio_files):
    soundscape_id = os.path.splitext(os.path.basename(audio_path))[0]

    # ── Load audio ────────────────────────────────────────────────
    t0 = time.perf_counter()
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    timing['audio'] += time.perf_counter() - t0

    total_samples = len(y)
    n_windows = int(np.ceil(total_samples / WINDOW_SAMPLES))

    # ── Predict each window with TTA ──────────────────────────────
    for win_idx in range(n_windows):
        center_start = win_idx * WINDOW_SAMPLES
        end_time_seconds = (win_idx + 1) * int(WINDOW_SECONDS)
        row_id = f'{soundscape_id}_{end_time_seconds}'

        t0 = time.perf_counter()
        probs = predict_window_tta(sessions, y, center_start,
                                   n_tta=n_tta_current, input_name=INPUT_NAME)
        timing['spec_plus_infer'] += time.perf_counter() - t0
        timing['windows'] += 1

        row = {'row_id': row_id}
        for sp_idx, sp in enumerate(SPECIES_LIST):
            row[sp] = float(probs[sp_idx])
        rows.append(row)

    # ── Time budget check ────────────────────────────────────────
    elapsed = time.time() - total_start
    rate = (file_idx + 1) / elapsed
    remaining = (len(audio_files) - file_idx - 1) / rate if rate > 0 else 0
    est_total = elapsed + remaining

    if est_total > TIME_BUDGET_SECONDS and n_tta_current > 1:
        old = n_tta_current
        n_tta_current = max(1, n_tta_current - 1)
        tta_downgrades.append({
            'soundscape_idx': file_idx + 1,
            'elapsed_min': elapsed / 60,
            'est_total_min': est_total / 60,
            'from_tta': old,
            'to_tta': n_tta_current,
        })
        print(f'  ⚠ TIME-BUDGET FALLBACK at soundscape {file_idx+1}/{len(audio_files)}: '
              f'est_total={est_total/60:.1f}min > budget={TIME_BUDGET_SECONDS/60:.0f}min. '
              f'Dropping TTA: {old}× → {n_tta_current}×')

    if (file_idx + 1) % 10 == 0 or file_idx == 0 or file_idx == len(audio_files) - 1:
        warn = '  ⚠' if est_total > TIME_WARN_SECONDS else ' '
        print(f'{warn} [{file_idx+1:>4}/{len(audio_files)}] '
              f'{soundscape_id}: {n_windows} windows | '
              f'TTA={n_tta_current}× | '
              f'Elapsed: {elapsed/60:.1f}min | '
              f'ETA: {remaining/60:.1f}min | '
              f'Total est: {est_total/60:.1f}min')

total_elapsed = time.time() - total_start
n_win = timing['windows']
print(f'\nDone! {len(audio_files)} soundscapes, {n_win} windows in '
      f'{total_elapsed:.1f}s ({total_elapsed/60:.1f}min)')
if n_win > 0:
    print(f'  Per window: audio={timing["audio"]/max(len(audio_files),1)*1000:.1f}ms/file, '
          f'spec+infer={timing["spec_plus_infer"]/n_win*1000:.1f}ms/window')

if tta_downgrades:
    print(f'\n⚠ TTA was downgraded {len(tta_downgrades)} time(s):')
    for d in tta_downgrades:
        print(f'  At soundscape {d["soundscape_idx"]}: '
              f'TTA {d["from_tta"]}× → {d["to_tta"]}× '
              f'(elapsed {d["elapsed_min"]:.1f}min, projected {d["est_total_min"]:.1f}min)')
else:
    print(f'\n✓ Completed with full {N_TTA_INITIAL}× TTA throughout.')


In [7]:
# ═══════════════════════════════════════════════════════════════════
# BUILD SUBMISSION
# ═══════════════════════════════════════════════════════════════════
expected_cols = list(sample_sub.columns)

if rows:
    submission = pd.DataFrame(rows)
    # Ensure exact column match with sample submission
    for col in expected_cols:
        if col not in submission.columns:
            submission[col] = 0.0
    submission = submission[expected_cols]
else:
    # No test files found — use sample_submission as skeleton
    submission = sample_sub.copy()
    print('⚠ No audio files found — using sample submission as placeholder')

# Validate
assert list(submission.columns) == expected_cols, 'Column mismatch!'
print(f'Submission shape: {submission.shape}')
print(f'Expected shape:   ({len(sample_sub)}, {len(expected_cols)})')
print(f'Columns match:    ✓')

# Quick sanity check
pred_vals = submission[SPECIES_LIST].values
print(f'\nPrediction stats:')
print(f'  Mean: {pred_vals.mean():.6f}')
print(f'  Std:  {pred_vals.std():.6f}')
print(f'  Min:  {pred_vals.min():.6f}')
print(f'  Max:  {pred_vals.max():.6f}')

submission.head()

⚠ No audio files found — using sample submission as placeholder
Submission shape: (3, 235)
Expected shape:   (3, 235)
Columns match:    ✓

Prediction stats:
  Mean: 0.004274
  Std:  0.000000
  Min:  0.004274
  Max:  0.004274


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


In [8]:
# ═══════════════════════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════════════════════
submission.to_csv('submission.csv', index=False)
print(f'  {submission.shape[0]} rows × {submission.shape[1]} columns')

  3 rows × 235 columns


In [9]:
pd.read_csv('/kaggle/working/submission.csv')

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
